# Семинар 08. Декораторы


## Цели

После семинара вы сможете:

- передавать функции как значения и создавать замыкания;
- писать обычные и параметризованные декораторы;
- сохранять метаданные функции с functools.wraps.

## Перед началом

Понадобятся функции высшего порядка, вложенные функции, *args, **kwargs и исключения.


В Python функции — объекты первого класса (`first-class citizens`): их можно присваивать переменным, передавать как аргументы и возвращать из других функций.



In [ ]:
def my_function():
    print("Hello, world")

other_function = my_function

other_function()

In [ ]:
%load_ext nb_mypy

from typing import Any, Callable


def function_creator(func: Callable[..., Any]) -> Callable[..., Any]:
    """
    Принимаем на вход функцию, а возвращаем другую функцию, которую сами только что сделали
    Может иметь отношение к исходной, а может и нет (порицается, но не исключено)
    """
    def function_wrapper(*args, **kwargs):
        print("Gotcha")
        return func(*args, **kwargs)
    
    return function_wrapper    

def f(x: int, y: int) -> int:
    return x + y

print(f(1, 2))

g = function_creator(f)

print(g(2, 3))
f = g
print(f(3, 4))

То же самое можно записать короче: выражение вида
```python
f = function_creator(f)
```
Удобно записывается так


In [ ]:
from typing import Any, Callable


def function_creator(func: Callable[..., Any]) -> Callable[..., Any]:
    def function_wrapper(*args, **kwargs):
        print("Gotcha")
        return func(*args, **kwargs)
    
    return function_wrapper

@function_creator
def my_function(a, b):
    return a + b

print(my_function(1, 2))  

Функция или другой вызываемый объект (`Callable`), который оборачивает другой вызываемый объект, называется **декоратором**.

Зачем это нужно:
Добавить логику, сопутствующую основной логике декорируемого `Callable`:
  * Логирование переданных параметров и результата
  * Авторизация и проверка разрешений
  * Повторное выполнение каких-то запросов в случае возникновения временных ошибок, когда безопасно пробовать еще раз
  * Кэширование ранее сделанных вычислений (мемоизация)
  * Регистрация обработчиков во веб-фреймворках и библиотеках для ботов
  * Встроенные конструкции Python: `@classmethod`, `@staticmethod`, `@property` и другие

In [ ]:
# Правила хорошего тона
from typing import Any, Callable
import functools
def my_decorator(func: Callable[[Any], Any]):
    # внутреннюю функцию называем `wrapper`
    # сигнатура у нее в общем случае `def wrapper(*args, **kwargs)`, потому что неизвестно, какая будет сигнатура у
    # декорируемой функции. 
    
    # @functools.wraps(func) нужно для того, чтобы не потерять информацию о декорируемой функции:
    # имя, документация, модуль и другие
    @functools.wraps(func)
    # крайне порицается добавление новых аргументов, которых не было в исходной.
    # extra_arg добавлять не надо!
    # Почему? потому что если вдруг появляется функция с тем же именем,
    #  то ожидается, что и вести себя она будет так же, как исходная
    def wrapper(*args, extra_arg=None, **kwargs) -> Any:
        print("Before")
        # здесь гипотетически мы можем вызвать декорируемую функцию не с переданными аргументами,
        # а с произвольными. Делать этого чаще всего не нужно и даже вредно
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper

@my_decorator
def f(x):
    print(x)

f(10)

В примере выше декоратор менял исходную функцию всегда одинаково и независимо ни от чего. Но что, если есть необходимость управлять этим поведением? 

In [ ]:

import functools

# здесь аргументы `retry_decorator` - параметры самого декоратора 
def retry_decorator(attempts: int):
    def decorator(func):
        # помним про волшебную функцию `wraps`
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # здесь есть доступ и к аргументам `retry_decorator`, и к аргументам `decorator` 
            # и к аргументам `func`
            if attempts < 1:
                raise ValueError("attempts must be positive")
            last_error = None
            for _ in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as error:
                    last_error = error
                    print(f"Exception occurred: {error}")
            raise RuntimeError("Maximum attempts reached") from last_error
        return wrapper
    # именно эту функцию вернем, а она, будучи вызванной с нужными аргументами, 
    # вернет подготовленный декоратор, который уже модифицирует исходную функцию
    return decorator
    
@retry_decorator(3)
def my_function():
    print("Hello, world")

@retry_decorator(3)
def my_function_with_args(arg1, arg2):
    print(f"Hello, {arg1} and {arg2}")

my_function_with_args("John", "Doe")

## Задание 1. Логирующий декоратор

Напишите декоратор, который при каждом вызове выводит имя функции, `args`, `kwargs` и результат. Если функция выбросила исключение, залогируйте его и пробросьте дальше.

```python
def log_call(func):
    ...
```

**Критерии проверки:** поддерживаются произвольные аргументы; исключения не подавляются; имя и документация исходной функции сохранены через `functools.wraps`.


## Задание 2. Параметризованный volkswagen-test декоратор

```python
def test_decorator(test_mode: bool = False):
    # test_mode=True: перехватить исключение и вернуть None
    # test_mode=False: полностью сохранить поведение исходной функции
    ...
```

**Критерии проверки:** обе ветви поведения покрыты тестами; успешный результат функции не меняется; метаданные исходной функции сохранены.
